In [1]:
from IPython.core.display import HTML
HTML("""
    <style>
    body { font-feature-settings: "liga" 0; }
    </style>
""")

# foreground.extend_process()
This procedure is meant to build a foreground fragment based on the inventory of its anchor.  If the 
fragment's anchor varies under different scenarios, then the process model should likewise vary.

In [1]:
# first we build out our catalog from blackbook
from antelope_foreground import ForegroundCatalog
from antelope import enum

from antelope_reports import QuickAndEasy



In [2]:
cat = ForegroundCatalog()
fg = cat.create_foreground('demo')
Q = QuickAndEasy(fg)

Loading JSON data from /data/GitHub/Antelope/core/antelope_core/archives/data/elcd_reference_quantities.json:
local.qdb: /data/GitHub/Antelope/core/antelope_core/archives/data/elcd_reference_quantities.json
local.qdb: Setting NSUUID (False) 77833297-6780-49bf-a61a-0cb707dce700
local.qdb: /data/GitHub/lca-tools/lcatools/qdb/data/elcd_reference_quantities.json
29 total quantity entities added (29 new)
6 total flow entities added (6 new)
QQQQQQQQQQQQQQQQQQ demo QQQQQQQQQQQQQQQQQQ
Found Antelope providers:
AntelopeMeta:_dev
antelope_core.providers:IlcdArchive
antelope_core.providers:IlcdLcia
antelope_core.providers:EcospoldV2Archive
antelope_core.providers:EcospoldV1Archive
antelope_core.providers:EcoinventLcia
antelope_core.providers:OpenLcaJsonLdArchive
antelope_core.providers:Traci21Factors
antelope_core.providers:XdbClient
antelope_core.providers:OpenLcaRefData
antelope_background.providers:TarjanBackground
antelope_background.providers:Background
antelope_foreground.providers:Antelope

Make a "heat from electricity" process with 90% efficiency

In [3]:
ncv = cat.get_canonical('net calorific value')

heat_elec = Q.new_link('Heat from electricity', ncv, 'Output')
heat_elec.observe(0.9)
heat_in_e = Q.new_link('Heat in from electricity', ncv, 'Input', parent=heat_elec, amount=1.0)


Naming fragment c4b39e05-6b55-4034-b541-8acd3ac2d8e9 -> heat_from_electricity


In [4]:
heat_elec.show_tree(True)

   -<--O   c4b39 [*    0.9 MJ] Heat from electricity
    [   1 unit] Heat from electricity
       | -<----: 5175b [       1 MJ] Heat in from electricity
       x 


Now we link the model to a couple of USLCI electricity processes

In [6]:
#cat.blackbook_guest('https://blackbook.localhost', verify='/data/GitHub/Antelope/blackbook/dev/certs/blackbook.localhost.pem')
#cat.get_blackbook_resources('lcacommons.uslci.fy24.q1.01', verify='/data/GitHub/Antelope/blackbook/dev/certs/xdb.localhost.pem')[0]
#q_us = cat.query('lcacommons.uslci')

elecs = enum(q_us.processes(name='electricity, at grid'))


GET https://xdb.localhost/lcacommons.uslci.fy24.q1.01/processes.. 200 [0.80 sec]
 [00] [lcacommons.uslci.fy24.q1.01] Electricity, at Grid, MRO, 2008 [Northern America]
 [01] [lcacommons.uslci.fy24.q1.01] Electricity, at Grid, RFC, 2008 [Northern America]
 [02] [lcacommons.uslci.fy24.q1.01] Electricity, at Grid, RFC, 2010 [Northern America]
 [03] [lcacommons.uslci.fy24.q1.01] Electricity, at Grid, TRE, 2010 [Northern America]
 [04] [lcacommons.uslci.fy24.q1.01] Electricity, at Grid, FRCC, 2008 [Northern America]
 [05] [lcacommons.uslci.fy24.q1.01] Electricity, at Grid, NPCC, 2010 [Northern America]
 [06] [lcacommons.uslci.fy24.q1.01] Electricity, at Grid, HICC, 2010 [Northern America]
 [07] [lcacommons.uslci.fy24.q1.01] Electricity, at Grid, NPCC, 2008 [Northern America]
 [08] [lcacommons.uslci.fy24.q1.01] Electricity, at Grid, FRCC, 2010 [Northern America]
 [09] [lcacommons.uslci.fy24.q1.01] Electricity, at Grid, TRE, 2008 [Northern America]
 [10] [lcacommons.uslci.fy24.q1.01] Electric

In [7]:
p = elecs[20]

In [18]:
from jwt import decode
def decode_token(_tok):
    return decode(_tok, 'a fish', options={'verify_signature': False})


In [8]:
fg.observe(heat_in_e, anchor_node=elecs[20])  # MRO
fg.observe(heat_in_e, anchor_node=elecs[13], scenario='WECC')  # WECC

GET https://xdb.localhost/lcacommons.uslci.fy24.q1.01/f6811440-ee37-11de-8a39-0800200c9a66.. 200 [0.05 sec]


'demo/5175b570-31a5-468a-9d66-cd1da8343e02'

In [9]:
heat_in_e.show()

(c4b39e0) -<- 5175b -<- -B*   [   1 MJ] Heat in from electricity 
LcFragment Entity (ref 5175b570-31a5-468a-9d66-cd1da8343e02)
origin: demo
reference: ( ** ref) -<- c4b39 -<- -O    [   1 MJ] Heat from electricity {heat_from_electricity}
     Name: Heat in from electricity
  Comment: 
StageName: Electricity, at Grid, MRO, 2010
Exchange values: 
              Cached: 1
            Observed: 1

Balance flow: False
Terminations: 
            Scenario  Termination
                None: -B*        [lcacommons.uslci.fy24.q1.01] Electricity, at Grid, MRO, 2010 [Northern America]
                WECC: -B*        [lcacommons.uslci.fy24.q1.01] Electricity, at Grid, WECC, 2010 [Northern America]


In [39]:
heat_in_e.show_tree()

   -<--B*  626a1 [       1 MJ] Heat in from electricity


In [11]:
cat.blackbook_authenticate('https://blackbook.localhost', 'scope3', 
                           verify='/data/GitHub/Antelope/blackbook/dev/certs/blackbook.localhost.pem')

POST https://blackbook.localhost/auth/token.. 200 [0.24 sec]
Welcome back to blackbook, hosted by ANTELOPE_AT_HOME.
Username: scope3, email: modeler@scope3consulting.com
Last login: Tue Sep  9 05:21:16 2025 
token expires in 36000 s


In [13]:
cat.refresh_xdb_tokens()

GET https://blackbook.localhost/origins/lcacommons.uslci.fy24.q1.01/token.. 200 [0.07 sec]
lcacommons.uslci.fy24.q1.01:background,v:basic,v:exchange,v:index,v
token expires in 999997 s


[LcResource(lcacommons.uslci.fy24.q1.01, dataSource=https://xdb.localhost/:XdbClient, ['index', 'basic', 'exchange', 'background'] [50] loaded  1 cfg)]

In [14]:
fg.extend_process(heat_in_e)

GET https://xdb.localhost/lcacommons.uslci.fy24.q1.01/c6fe7799-8cb7-32e6-9bd9-a3d5564c218f/06581fb2-1de0-3e78-8298-f37605dea142/dependencies.. 200 [0.04 sec]
GET https://xdb.localhost/lcacommons.uslci.fy24.q1.01/c6fe7799-8cb7-32e6-9bd9-a3d5564c218f/06581fb2-1de0-3e78-8298-f37605dea142/cutoffs.. 200 [0.01 sec]
GET https://xdb.localhost/lcacommons.uslci.fy24.q1.01/e391a117-69ae-3550-987a-8f28a1444eb9.. 200 [0.00 sec]
GET https://xdb.localhost/lcacommons.uslci.fy24.q1.01/contexts/2211: Electric Power Generation, Transmission and Distribution.. 200 [0.00 sec]
GET https://xdb.localhost/lcacommons.uslci.fy24.q1.01/contexts/22: Utilities.. 200 [0.00 sec]
GET https://xdb.localhost/lcacommons.uslci.fy24.q1.01/contexts/Technosphere Flows.. 200 [0.00 sec]
GET https://xdb.localhost/lcacommons.uslci.fy24.q1.01/5b26dd62-5ab0-3822-b39c-6aa187efc9e5.. 200 [0.00 sec]
GET https://xdb.localhost/lcacommons.uslci.fy24.q1.01/2b091e0d-d0de-31a1-89db-3f51a9c5cf12.. 200 [0.00 sec]
GET https://xdb.localhost/lca

In [15]:
heat_elec.show_tree('MRO')

   -<--O   c4b39 [*    0.9 MJ] Heat from electricity
    [   1 unit] Heat from electricity
       |        Stage: Electricity, at Grid, MRO, 2010
       | -<--*   5175b [       1 MJ] Heat in from electricity
       |  [   1 MJ] Heat in from electricity
       |     |        Stage: CUTOFF Flows
       |     | -<----: 1c248 [  0.0042 MJ] Electricity, biomass,liquid, unspecified, at power plant
       |     | -<----: 8c089 [ 0.00202 MJ] Electricity, biomass, gas, landfill, at power plant
       |     | -<----: c1436 [   0.187 MJ] Electricity, hydropower, at power plant, unspecified
       |     | -<----: 31e2c [ 0.00668 MJ] Electricity, biomass, solid, agriculture by-products, at power plant
       |     | -<----: d52da [   7e-06 MJ] Electricity, solar, unspecified, at power plant
       |     | -<----: 1c4d0 [  0.0016 MJ] Electricity, other fuels, unspecified, at power plant
       |     | -<----: 3567f [ 0.000136 MJ] Electricity, other gases, unspecified,  at power plant
       |     | 

Get an LCIA method

In [16]:
cat.get_blackbook_resources('lcia.traci.2.1', verify='/data/GitHub/Antelope/blackbook/dev/certs/xdb.localhost.pem')
gwp = next(cat.query('lcia.traci').lcia_methods(name='Global Warming'))


GET https://blackbook.localhost/origins/lcia.traci.2.1/resource.. 200 [0.06 sec]
QQQQQQQQQQQQQQQQQQ lcia.traci.2.1 QQQQQQQQQQQQQQQQQQ
GET https://xdb.localhost/origins.. 200 [0.01 sec]
lcia.traci.2.1: https://xdb.localhost/
lcia.traci.2.1: Setting NSUUID (False) None
GET https://xdb.localhost/lcia.traci.2.1/config.. 200 [0.01 sec]
Applying configuration to XdbClient with 0 entities at https://xdb.localhost/
Applying context hint lcia.traci.2.1:water => to water
Applying context hint lcia.traci.2.1:air => to air
GET https://xdb.localhost/lcia.traci.2.1/lcia_methods.. 200 [0.00 sec]


In [17]:
inv=enum(p.inventory())

GET https://xdb.localhost/lcacommons.uslci.fy24.q1.01/c6fe7799-8cb7-32e6-9bd9-a3d5564c218f/inventory.. 200 [0.01 sec]
 [00] [ Electricity, at Grid, MRO, 2010 [Northern America] ]*==>  3.6 (MJ) Electricity, at grid 
 [01] [ Electricity, at Grid, MRO, 2010 [Northern America] ] <--# 0.00353 (MJ) Electricity, diesel, at power plant 
 [02] [ Electricity, at Grid, MRO, 2010 [Northern America] ] <--# 0.469 (MJ) Electricity, nuclear, at power plant 
 [03] [ Electricity, at Grid, MRO, 2010 [Northern America] ] <--# 0.000108 (MJ) Electricity, residual fuel oil, at power plant 
 [04] [ Electricity, at Grid, MRO, 2010 [Northern America] ] <--# 0.538 (MJ) Electricity, lignite coal, at power plant 
 [05] [ Electricity, at Grid, MRO, 2010 [Northern America] ] <--# 0.0406 (MJ) Electricity, natural gas, at power plant 
 [06] [ Electricity, at Grid, MRO, 2010 [Northern America] ] <--# 1.49 (MJ) Electricity, bituminous coal, at power plant 
 [07] [ Electricity, at Grid, MRO, 2010 [Northern America] ] <--

In [19]:
import pandas

In [20]:
df = pandas.DataFrame({k: (heat_elec.fragment_lcia(gwp, scenario=k).components_as_series) for k in ('MRO', 'WECC')})

POST https://xdb.localhost/lcacommons.uslci.fy24.q1.01/c6fe7799-8cb7-32e6-9bd9-a3d5564c218f/06581fb2-1de0-3e78-8298-f37605dea142/lcia/Global Warming Air.. 200 [0.01 sec]
GET https://xdb.localhost/lcacommons.uslci.fy24.q1.01/e391a117-69ae-3550-987a-8f28a1444eb9/f7d232cc-70aa-3669-a4e4-d1083d248e69/lcia/Global Warming Air.. 200 [0.67 sec]
GET https://xdb.localhost/lcacommons.uslci.fy24.q1.01/contexts/to air.. 200 [0.00 sec]
GET https://xdb.localhost/lcacommons.uslci.fy24.q1.01/contexts/urban.. 200 [0.00 sec]
GET https://xdb.localhost/lcacommons.uslci.fy24.q1.01/contexts/troposphere.. 200 [0.00 sec]
GET https://xdb.localhost/lcacommons.uslci.fy24.q1.01/contexts/rural.. 200 [0.00 sec]
GET https://xdb.localhost/lcacommons.uslci.fy24.q1.01/contexts/stratosphere.. 200 [0.00 sec]
GET https://xdb.localhost/lcacommons.uslci.fy24.q1.01/5b26dd62-5ab0-3822-b39c-6aa187efc9e5/a9f6971f-58a0-30b0-af5c-a36d11fe487f/lcia/Global Warming Air.. 200 [0.06 sec]
GET https://xdb.localhost/lcacommons.uslci.fy24.

In [21]:
df

,MRO,WECC
"Electricity, residual fuel oil, at power plant [Northern America]",0.000009,NaN
"Electricity, lignite coal, at power plant [Northern America]",0.049494,NaN
"Electricity, nuclear, at power plant [Northern America]",0.000311,0.000182
"Electricity, diesel, at power plant [Northern America]",0.000341,0.014682
"Electricity, natural gas, at power plant [Northern America]",0.002284,0.018077
"Electricity, bituminous coal, at power plant [Northern America]",0.124681,0.076038


In [22]:
_=enum(heat_elec.cutoffs('WECC'))

 [00] [ heat_from_electricity ] <--  0.224 (MJ) Electricity, hydropower, at power plant, unspecified  (cutoff)
 [01] [ heat_from_electricity ] <--  0.0196 (MJ) Electricity, geothermal, unspecified  (cutoff)
 [02] [ heat_from_electricity ] <--  0.0195 (MJ) Electricity, at wind power plant, unspecified  (cutoff)
 [03] [ heat_from_electricity ] <--  0.0193 (MJ) Electricity, petroleum coke, at power plant  (cutoff)
 [04] [ heat_from_electricity ] <--  0.00884 (MJ) Electricity, biomass, solid, agriculture by-products, at power plant  (cutoff)
 [05] [ heat_from_electricity ] <--  0.00211 (MJ) Electricity, other gases, unspecified,  at power plant  (cutoff)
 [06] [ heat_from_electricity ] <--  0.00194 (MJ) Electricity, biomass, gas, landfill, at power plant  (cutoff)
 [07] [ heat_from_electricity ] <--  0.00109 (MJ) Electricity, other fuels, unspecified, at power plant  (cutoff)
 [08] [ heat_from_electricity ] <--  0.000822 (MJ) Electricity, biomass,liquid, unspecified, at power plant  (cutof